<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Histogram**


Estimated time needed: **45** minutes


In this lab, you will focus on the visualization of data. The dataset will be provided through an RDBMS, and you will need to use SQL queries to extract the required data.


## Objectives


In this lab, you will perform the following:


- Visualize the distribution of data using histograms.

- Visualize relationships between features.

- Explore data composition and comparisons.


## Demo: Working with database


#### Download the database file.


In [ ]:
#wget -O survey-data.sqlite https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QR9YeprUYhOoLafzlLspAw/survey-results-public.sqlite

#### Install the required libraries and import them


In [ ]:
#pip install pandas

In [ ]:
#pip install matplotlib

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

#### Connect to the SQLite database


In [ ]:
conn = sqlite3.connect('survey-data.sqlite')

## Demo: Basic SQL queries

**Demo 1: Count the number of rows in the table**


In [ ]:
QUERY = "SELECT COUNT(*) FROM main"
df = pd.read_sql_query(QUERY, conn)
print(df)


**Demo 2: List all tables**


In [ ]:
QUERY = """
SELECT name as Table_Name 
FROM sqlite_master 
WHERE type = 'table'
"""
pd.read_sql_query(QUERY, conn)


**Demo 3: Group data by age**


In [ ]:
QUERY = """
SELECT Age, COUNT(*) as count 
FROM main 
GROUP BY Age 
ORDER BY Age
"""
df_age = pd.read_sql_query(QUERY, conn)
print(df_age)


## Hands-on Lab: Visualizing Data with Histograms


### 1. Visualizing the distribution of data (Histograms)


**1.1 Histogram of `CompTotal` (Total Compensation)**


Objective: Plot a histogram of `CompTotal` to visualize the distribution of respondents' total compensation.


In [ ]:
QUERY = """
SELECT CompTotal
FROM main
WHERE CompTotal IS NOT NULL AND CompTotal > 0 AND CompTotal <= 500000
"""
df = pd.read_sql_query(QUERY, conn)

plt.figure(figsize=(10, 6))
plt.hist(df['CompTotal'], bins=50, color='steelblue', edgecolor='black', alpha=0.8)
plt.title('Distribution of Total Compensation (CompTotal)', fontsize=14)
plt.xlabel('Total Compensation (USD)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**1.2 Histogram of YearsCodePro (Years of Professional Coding Experience)**


Objective: Plot a histogram of `YearsCodePro` to analyze the distribution of coding experience among respondents.


In [ ]:
QUERY = """
SELECT
  CASE
    WHEN TRIM(YearsCodePro) = 'Less than 1 year' THEN 0
    WHEN TRIM(YearsCodePro) = 'More than 50 years' THEN 51
    ELSE CAST(TRIM(YearsCodePro) AS INTEGER)
  END AS YearsCodeProNumeric
FROM main
WHERE YearsCodePro IS NOT NULL AND TRIM(YearsCodePro) != ''
"""
df = pd.read_sql_query(QUERY, conn)

plt.figure(figsize=(10, 6))
plt.hist(df['YearsCodeProNumeric'], bins=range(0, 54, 2), color='seagreen', edgecolor='black', alpha=0.8)
plt.title('Distribution of Professional Coding Experience (YearsCodePro)', fontsize=14)
plt.xlabel('Years of Professional Coding Experience', fontsize=12)
plt.ylabel('Number of Respondents', fontsize=12)
plt.xticks(range(0, 52, 5))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 2. Visualizing Relationships in Data


**2.1 Histogram Comparison of `CompTotal` by `Age` Group**


Objective: Use histograms to compare the distribution of CompTotal across different Age groups.


In [ ]:
QUERY = """
SELECT Age, CompTotal
FROM main
WHERE CompTotal IS NOT NULL AND CompTotal > 0 AND CompTotal <= 500000
  AND Age IS NOT NULL AND Age NOT IN ('Prefer not to say')
"""
df = pd.read_sql_query(QUERY, conn)

age_order = ['Under 18 years old', '18-24 years old', '25-34 years old',
             '35-44 years old', '45-54 years old', '55-64 years old',
             '65 years or older']

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for i, age in enumerate(age_order):
    subset = df[df['Age'] == age]['CompTotal']
    axes[i].hist(subset, bins=40, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Age: {age} (n={len(subset)})', fontsize=11)
    axes[i].set_xlabel('CompTotal (USD)')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(axis='y', alpha=0.3)

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

**2.2 Histogram of TimeSearching for Different Age Groups**


Objective: Use histograms to explore the distribution of `TimeSearching` (time spent searching for information) for respondents across different age groups.


In [ ]:
QUERY = """
SELECT Age, TimeSearching
FROM main
WHERE TimeSearching IS NOT NULL AND TRIM(TimeSearching) != ''
  AND Age IS NOT NULL AND Age NOT IN ('Prefer not to say')
"""
df = pd.read_sql_query(QUERY, conn)

time_order = ['Less than 15 minutes a day', '15-30 minutes a day',
              '30-60 minutes a day', '60-120 minutes a day',
              'Over 120 minutes a day']
age_order = ['Under 18 years old', '18-24 years old', '25-34 years old',
             '35-44 years old', '45-54 years old', '55-64 years old',
             '65 years or older']


ct = pd.crosstab(df['Age'], df['TimeSearching'])[time_order]
ct = ct.reindex(age_order, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
bottom = np.zeros(len(age_order))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(time_order)))
for i, t in enumerate(time_order):
    ax.barh(age_order, ct[t], left=bottom, label=t, color=colors[i], edgecolor='white')
    bottom += ct[t].values
ax.set_xlabel('Number of Respondents')
ax.set_title('TimeSearching Distribution by Age Group')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)

ax = axes[1]
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(age_order))
for i, t in enumerate(time_order):
    ax.barh(age_order, ct_pct[t], left=bottom, label=t, color=colors[i], edgecolor='white')
    bottom += ct_pct[t].values
ax.set_xlabel('Percentage (%)')
ax.set_title('TimeSearching Distribution by Age Group (Normalized)')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

### 3. Visualizing the Composition of Data


**3.1 Histogram of Most Desired Databases (`DatabaseWantToWorkWith`)**


Objective: Visualize the most desired databases for future learning using a histogram of the top 5 databases.


In [ ]:
QUERY = """
SELECT DatabaseWantToWorkWith
FROM main
WHERE DatabaseWantToWorkWith IS NOT NULL AND TRIM(DatabaseWantToWorkWith) != ''
"""
df = pd.read_sql_query(QUERY, conn)

all_dbs = []
for val in df['DatabaseWantToWorkWith']:
    for db in val.split(';'):
        db = db.strip()
        if db:
            all_dbs.append(db)

db_counts = pd.Series(all_dbs).value_counts().head(5)

plt.figure(figsize=(10, 6))
plt.bar(db_counts.index, db_counts.values, color='coral', edgecolor='black', alpha=0.8)
plt.title('Top 5 Most Desired Databases to Learn', fontsize=14)
plt.xlabel('Database', fontsize=12)
plt.ylabel('Number of Respondents', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**3.2 Histogram of Preferred Work Locations (`RemoteWork`)**


Objective: Use a histogram to explore the distribution of preferred work arrangements (`remote work`).


In [ ]:
QUERY = """
SELECT RemoteWork, COUNT(*) as count
FROM main
WHERE RemoteWork IS NOT NULL AND TRIM(RemoteWork) != ''
GROUP BY RemoteWork
ORDER BY count DESC
"""
df = pd.read_sql_query(QUERY, conn)

plt.figure(figsize=(8, 6))
plt.bar(df['RemoteWork'], df['count'], color=['#2ecc71', '#3498db', '#e74c3c', '#95a5a6'],
        edgecolor='black', alpha=0.8)
plt.title('Distribution of Preferred Work Arrangements (RemoteWork)', fontsize=14)
plt.xlabel('Work Arrangement', fontsize=12)
plt.ylabel('Number of Respondents', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 4. Visualizing Comparison of Data


**4.1 Histogram of Median CompTotal for Ages 45 to 60**


Objective: Plot the histogram for `CompTotal` within the age group 45 to 60 to analyze compensation distribution among mid-career respondents.


In [ ]:
QUERY = """
SELECT Age, CompTotal
FROM main
WHERE CompTotal IS NOT NULL AND CompTotal > 0 AND CompTotal <= 500000
  AND Age IN ('45-54 years old', '55-64 years old')
"""
df = pd.read_sql_query(QUERY, conn)

plt.figure(figsize=(10, 6))
plt.hist(df['CompTotal'], bins=50, color='purple', edgecolor='black', alpha=0.8)
plt.title('CompTotal Distribution (Ages 45-64)', fontsize=14)
plt.xlabel('Total Compensation (USD)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
median_val = df['CompTotal'].median()
plt.axvline(median_val, color='red', linestyle='--', linewidth=2,
            label=f'Median = ${median_val:,.0f}')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**4.2 Histogram of Job Satisfaction (`JobSat`) by YearsCodePro**


Objective: Plot the histogram for `JobSat` scores based on respondents' years of professional coding experience.


In [ ]:
QUERY = """
SELECT
  CASE
    WHEN TRIM(YearsCodePro) = 'Less than 1 year' THEN 0
    WHEN TRIM(YearsCodePro) = 'More than 50 years' THEN 51
    ELSE CAST(TRIM(YearsCodePro) AS INTEGER)
  END AS YearsCodeProNumeric,
  JobSat
FROM main
WHERE YearsCodePro IS NOT NULL AND TRIM(YearsCodePro) != ''
  AND JobSat IS NOT NULL
"""
df = pd.read_sql_query(QUERY, conn)

df['ExperienceGroup'] = pd.cut(df['YearsCodeProNumeric'],
                               bins=[-1, 2, 5, 10, 20, 100],
                               labels=['0-2 yrs', '3-5 yrs', '6-10 yrs', '11-20 yrs', '20+ yrs'])

fig, axes = plt.subplots(1, 5, figsize=(18, 5), sharey=True)

for i, (label, group) in enumerate(df.groupby('ExperienceGroup', observed=True)):
    axes[i].hist(group['JobSat'], bins=range(0, 12), color='teal',
                 edgecolor='black', alpha=0.8, align='left')
    axes[i].set_title(f'{label} (n={len(group)})', fontsize=10)
    axes[i].set_xlabel('JobSat Score')
    axes[i].set_xticks(range(0, 11))
    axes[i].grid(axis='y', alpha=0.3)

axes[0].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

### Final step: Close the database connection


Once you've completed the lab, make sure to close the connection to the SQLite database:



In [ ]:
conn.close()

### Summary


In this lab, you used histograms to visualize various aspects of the dataset, focusing on:

- Distribution of compensation, coding experience, and work hours.

- Relationships in compensation across age groups and work status.

- Composition of data by desired databases and work environments.

- Comparisons of job satisfaction across years of experience.

Histograms helped reveal patterns and distributions in the data, enhancing your understanding of developer demographics and preferences.


## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


Copyright © IBM Corporation. All rights reserved.
